# 02-10. 기간이동 계산(rolling)

## rolling()
: 현재 열에 대하여 일정 크기의 창(window)을 이동시키며, 그 window 안의 값을 추가 메서드를 통해 계산하는 메서드이다.

### 사용법
```python
df.rolling(window, min_periods=None, center=False, win_type=None, on=None, closed=None, method='single').추가메서드()
```
- window : 계산할 창(window)의 크기이다. 기본적으로 각 열에서 최근 행의 수를 의미한다. 시간 문자열도 사용할 수 있다.
- min_periods : 계산할 최소 크기(기간)이다. window 안의 값의 수가 min_periods보다 작을 경우 NaN을 출력한다. 기본적으로 window 크기와 동일하다.
- center : { True / False } 레이블을 window의 중간에 둘지 여부이다. 기본값은 False로 레이블이 창 우측에 위치한다.
- win_type : { 'triang' / 'gaussian' / ... } 가중치를 넣어 계산할 경우 계산 방식이다. 때에 따라 연산 메서드에 추가 인수를 지정해야 할 수도 있다.
- on : 시계열 인덱스나, 시계열과 유사한 열이 있을 경우 이 열을 기준으로 rolling을 수행할 수 있다.
- axis : 현재 pandas의 rolling()에서는 지원하지 않는다. 행 방향 계산은 `df.T.rolling(...).sum().T`처럼 전치해서 수행한다.
- closed : { 'left' / 'right' / 'both' / 'neither' } window가 닫히는 방향이다.
- method : { 'single' / 'table' } numba 엔진을 사용할 때 계산 단위를 지정한다. 'single'은 열별 계산이고, 'table'은 전체 DataFrame을 함께 계산한다.
    - 'table'은 추가 연산 메서드에서 `engine='numba'`를 지정해야 사용할 수 있다.

In [10]:
import numpy as np
import pandas as pd

In [11]:
# 6 x 2 DataFrame 생성
period = pd.period_range(start = '2022-01-13 00:00:00', end = '2022-01-13 02:30:00', freq = '30min')
data = {'col1': [1, 2, 3, 4, 5, 6], 'col2': period}
idx = ['row1', 'row2', 'row3', 'row4', 'row5', 'row6']

df = pd.DataFrame(data = data, index = idx)

print(df)

      col1              col2
row1     1  2022-01-13 00:00
row2     2  2022-01-13 00:30
row3     3  2022-01-13 01:00
row4     4  2022-01-13 01:30
row5     5  2022-01-13 02:00
row6     6  2022-01-13 02:30


In [12]:
# 기본적인 사용법 : window 크기지정 -> 현재 행 이전으로 window 크기만큼의 행을 가져와서 연산 수행
print(df[['col1']].rolling(window = 3).sum())

      col1
row1   NaN
row2   NaN
row3   6.0
row4   9.0
row5  12.0
row6  15.0


In [13]:
# closed 인수 사용 : closed 인수는 window의 시작과 끝을 포함할지 여부를 지정하는 인수이다. 

# 기본값은 'right'로, window의 끝을 포함한다.
print(df[['col1']].rolling(window = 3, closed = 'left').sum())

# closed='right'로 지정하면 window의 끝을 포함하고 시작을 포함하지 않음
print(df[['col1']].rolling(window = 3, closed = 'right').sum())

# closed='both'로 지정하면 window의 시작과 끝을 모두 포함
print(df[['col1']].rolling(window = 3, closed = 'both').sum())

# closed='neither'로 지정하면 window의 시작과 끝을 모두 포함하지 않음
print(df[['col1']].rolling(window = 3, closed = 'neither').sum())

      col1
row1   NaN
row2   NaN
row3   NaN
row4   6.0
row5   9.0
row6  12.0
      col1
row1   NaN
row2   NaN
row3   6.0
row4   9.0
row5  12.0
row6  15.0
      col1
row1   NaN
row2   NaN
row3   6.0
row4  10.0
row5  14.0
row6  18.0
      col1
row1   NaN
row2   NaN
row3   NaN
row4   NaN
row5   NaN
row6   NaN


In [14]:
# center 인수 사용 : center 인수는 window의 중심을 기준으로 연산을 수행할지 여부를 지정하는 인수이다.
# center=True로 지정하면 window의 중심을 기준으로 연산을 수행하고, center=False로 지정하면 window의 끝을 기준으로 연산을 수행한다.
print(df[['col1']].rolling(window = 3, center = True).sum())


      col1
row1   NaN
row2   6.0
row3   9.0
row4  12.0
row5  15.0
row6   NaN


In [15]:
# win_type 인수의 사용
print(df['col1'].rolling(window = 3, win_type = 'triang').sum()) # 삼각형 가중치

print(df['col1'].rolling(window = 3, win_type = 'gaussian').sum(std = 1)) # 가우시안 가중치

row1     NaN
row2     NaN
row3     4.0
row4     6.0
row5     8.0
row6    10.0
Name: col1, dtype: float64
row1          NaN
row2          NaN
row3     4.426123
row4     6.639184
row5     8.852245
row6    11.065307
Name: col1, dtype: float64


In [16]:
# on 인수의 사용 (on = 'col2'로 지정하면, col2 열의 값들을 기준으로 window를 설정하여 연산 수행)
print(df.rolling(window = '1h', on = 'col2').sum())


      col1              col2
row1   1.0  2022-01-13 00:00
row2   3.0  2022-01-13 00:30
row3   5.0  2022-01-13 01:00
row4   7.0  2022-01-13 01:30
row5   9.0  2022-01-13 02:00
row6  11.0  2022-01-13 02:30
